In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
GOLD_FATO_PATH   = "workspace.case_spark_cvm.gold_fato_diario"
NOME_TABELA      = f"gold_cubo_risco" 
GOLD_PATH        = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC        = int(datetime.now().strftime("%Y%m%d"))

### 1. gold_fato_diario

In [0]:
gold_fato_diario  = spark.read.table(GOLD_FATO_PATH)

In [0]:
display(gold_fato_diario)

### 2. Pegar Ultima Data do Drawdown

In [0]:
window_drawdown = Window.partitionBy("cnpj_fundo_classe").orderBy(f.col("drawdown_maximo_historico").asc(), f.col("dt_comptc").desc())

df_drawdown = gold_fato_diario\
    .withColumn("rn", f.row_number().over(window_drawdown))\
    .filter(f.col("rn") ==  1)\
    .select("cnpj_fundo_classe", f.col("dt_comptc").alias("data_drawdown_maximo"))

In [0]:
gold_fato_diario = gold_fato_diario\
    .join(
        df_drawdown,
        'cnpj_fundo_classe',
        "left"
    )

### 4. Pegando a Ultima Data do Fundo e Selecionando as Colunas de Risco

In [0]:
window_ultima_foto = Window.partitionBy("cnpj_fundo_classe").orderBy(f.col("dt_comptc").desc())

gold_cubo_risco = gold_fato_diario\
    .withColumn("rn_foto", f.row_number().over(window_ultima_foto)) \
    .filter(f.col("rn_foto") == 1) \
    .withColumn(
        "classificacao_risco",
        f.when(f.col("volatilidade_252d") < 0.02, "BAIXO")
         .when((f.col("volatilidade_252d") >= 0.02) & (f.col("volatilidade_252d") <= 0.10), "MÉDIO")
         .otherwise("ALTO")
    )\
    .select(
        "cnpj_fundo_classe",
        f.col("dt_comptc").alias("dt_referencia"),
        "volatilidade_21d",
        "volatilidade_63d",
        "volatilidade_252d",
        "drawdown_maximo_252d",
        "drawdown_maximo_historico",
        "data_drawdown_maximo",
        "var_95_252d",
        "classificacao_risco"
    )

In [0]:

log.info(f"Iniciando a escrita da dimensão unificada em: {GOLD_PATH}") 

PipelineConfig.gravar_cubo_gold(
    spark=spark,
    df_cubo=gold_cubo_risco,
    tabela_destino=GOLD_PATH,
    zorder_cols=["cnpj_fundo_classe", "classificacao_risco"],
)


log.info(f"Processamento da {GOLD_PATH} concluído com sucesso!")

In [0]:
display(gold_cubo_risco)